**Step 1: Identify the Task**

The task is to design and implement a **Knowledge-Based Agent (KBA)** for an **autonomous self-driving fuel trailer** operating across Kenya's Highway Network from an initial state (**Nairobi**) to a goal state (**Kisumu**). At every decision point, the agent combines:
* **Route & Distance Knowledge (from Practical Exercise 3):** Directed highway connections, kilometre distances, and abstracted search tree path costs $g(n)$.
* **Real-Time Percepts & Safety Rules:** Traffic light signals, live weather conditions along the corridor, road blockages, and fuel-tanker sensor telemetry (vapour/leak detection, roll angle, tank temperature, and tyre pressure).

**Goal:** Make safe, compliant, fuel-efficient, and optimal navigation decisions from Nairobi to Kisumu with zero crashes, rollovers, fuel leaks, spills, or fires.

### PEAS & Environment Summary
* **Performance Measure:** Safety of people, truck, and environment (no crashes, rollovers, leaks, spills, or fires); intact on-time fuel delivery; minimum safe trip distance; compliance with traffic and dangerous-goods transport rules.
* **Environment:** Kenya Highway Network (Nairobi, Nakuru, Kericho, Eldoret, Kakamega, Kisii, Kisumu), traffic signals, weather/road conditions (`Clear`, `Rain`, `Fog`), fuel tank load, and security/road hazards — classified as **Partially Observable, Stochastic, Sequential, Dynamic, and Multi-Agent**.
* **Actuators:** Steering, accelerator, brakes, gears, hazard lights, horn, emergency fuel shut-off valves, fire suppression system, and automatic emergency/dispatcher alert.
* **Sensors:** Front/rear/side cameras, GPS, radar/lidar, speedometer, fuel level/temperature sensors, leak & vapour detector, tank/tyre pressure sensors, load/roll-angle sensor, and operator keyboard input.

**Step 2: Assemble Relevant Knowledge**

The autonomous trailer's Knowledge Base (KB) integrates four core knowledge sources:
1. **Kenya Highway Network Directed Graph & Distances (Practical Exercise 3):**
   * `Nairobi <-> Nakuru` (161 km)
   * `Nakuru <-> Kericho` (117 km)
   * `Nakuru <-> Eldoret` (156 km)
   * `Nakuru -> Kisumu` (178 km)
   * `Kericho <-> Kisumu` (84 km)
   * `Kericho <-> Kisii` (104 km)
   * `Kisii -> Kisumu` (136 km)
   * `Eldoret -> Kisumu` (138 km)
   * `Eldoret <-> Kakamega` (106 km)
   * `Kakamega <-> Kisumu` (81 km)
2. **Abstracted Search Tree & Cumulative Path Costs $g(n)$ (Practical Exercise 3):**
   * `Nairobi -> Nakuru -> Kisumu`: $161 + 178 = 339\text{ km}$ *(Optimal path)*
   * `Nairobi -> Nakuru -> Kericho -> Kisumu`: $161 + 117 + 84 = 362\text{ km}$
   * `Nairobi -> Nakuru -> Eldoret -> Kisumu`: $161 + 156 + 138 = 455\text{ km}$
   * `Nairobi -> Nakuru -> Eldoret -> Kakamega -> Kisumu`: $161 + 156 + 106 + 81 = 504\text{ km}$
   * `Nairobi -> Nakuru -> Kericho -> Kisii -> Kisumu`: $161 + 117 + 104 + 136 = 518\text{ km}$
3. **Traffic Signal & Dangerous-Goods Driving Rules:** Standard signal actions (`Red`, `Yellow`, `Green`, `Flashing`, `Blinking`) paired with heavy fuel-tanker braking and speed limits.
4. **Weather & Tanker Telemetry Safety Rules:** Threshold rules mapping weather (`Clear`, `Rain`, `Fog`) and sensor states (`Normal`, `VapourLeak`, `HighRollAngle`, `HighTankTemp`, `LowTyrePressure`) to safe actuator commands.

```
API / Data Retrieval Declaration:
- Permitted Public API Used: Open-Meteo Weather Forecast API (https://api.open-meteo.com/v1/forecast)
  Free, open-access meteorological API (no API key required) queried using the GPS coordinates of the
  7 Kenyan towns in our Practical Exercise 3 network to enrich the KB with live weather states.
- Repository Companion Script: 'fetch_route_kb.py' (uploaded in the same 'Knowledge-Engineering'
  GitHub repository as 'Autonomous_Trailer.ipynb'). An automatic offline fallback is also built into
  Step 4 so the notebook runs cleanly without internet access.
```

**Step 3: Define the Vocabulary**

* **Predicates (represented in Python as structured dictionaries, tuples, and logical query functions):**
  * `Connected(town_a, town_b, distance_km)` — Directed highway link from `town_a` to `town_b` with step cost in kilometres.
  * `At(trailer, town)` — Current town node occupied by the trailer.
  * `Goal(town)` — Target destination town (`Kisumu`).
  * `PathCost(route, total_km)` — Cumulative path cost $g(n)$ along a sequence of towns.
  * `OptimalRoute(start, goal, route)` — Minimum-cost safe path from `start` to `goal`.
  * `Light(color)` / `TrafficAction(color, action)` — Observed traffic light state and mandated trailer response.
  * `Weather(town, condition)` — Observed or API-fetched weather state at a given town.
  * `SensorAlert(sensor_state, action)` — Tanker telemetry state and emergency/cautionary actuator response.
  * `Blocked(town_a, town_b)` — Dynamic road closure or hazard flag on a highway segment.

* **Constants:**
  * **Towns (Nodes):** `Nairobi`, `Nakuru`, `Kericho`, `Eldoret`, `Kakamega`, `Kisii`, `Kisumu`
  * **Traffic Lights:** `Red`, `Yellow`, `Green`, `Flashing`, `Blinking`
  * **Weather Conditions:** `Clear`, `Rain`, `Fog`
  * **Tanker Sensor States:** `Normal`, `VapourLeak`, `HighRollAngle`, `HighTankTemp`, `LowTyrePressure`
  * **Primary Actuator Responses:** `Proceed`, `SlowDown`, `Stop`, `Reroute`, `EmergencyShutOff`

**Step 4: Encode General Knowledge about the Domain**

```
Explicit declaration of the vocabulary, Practical Exercise 3 directed graph & abstracted search tree,
domain safety rules, and the Open-Meteo API weather loader declared in Step 2.
```

In [ ]:
import json
import heapq
import urllib.request
import urllib.error


TOWNS = ("Nairobi", "Nakuru", "Kericho", "Eldoret", "Kakamega", "Kisii", "Kisumu")
LIGHT_STATES = ("Red", "Yellow", "Green", "Flashing", "Blinking")
WEATHER_STATES = ("Clear", "Rain", "Fog")
SENSOR_STATES = ("Normal", "VapourLeak", "HighRollAngle", "HighTankTemp", "LowTyrePressure")

# GPS coordinates for the 7 towns in Kenya's Highway Network (used by Open-Meteo API)
TOWN_COORDINATES = {
    "Nairobi": (-1.2864, 36.8172),
    "Nakuru": (-0.3031, 36.0800),
    "Kericho": (-0.3677, 35.2831),
    "Eldoret": (0.5143, 35.2698),
    "Kakamega": (0.2827, 34.7519),
    "Kisii": (-0.6817, 34.7667),
    "Kisumu": (-0.0917, 34.7680),
}


HIGHWAY_GRAPH = {
    "Nairobi": {"Nakuru": 161},
    "Nakuru": {"Nairobi": 161, "Kericho": 117, "Eldoret": 156, "Kisumu": 178},
    "Kericho": {"Nakuru": 117, "Kisumu": 84, "Kisii": 104},
    "Kisii": {"Kericho": 104, "Kisumu": 136},
    "Eldoret": {"Nakuru": 156, "Kisumu": 138, "Kakamega": 106},
    "Kakamega": {"Eldoret": 106, "Kisumu": 81},
    "Kisumu": {"Kericho": 84, "Kakamega": 81},
}


PE3_SEARCH_TREE_ROUTES = [
    (["Nairobi", "Nakuru", "Kisumu"], 339),
    (["Nairobi", "Nakuru", "Kericho", "Kisumu"], 362),
    (["Nairobi", "Nakuru", "Eldoret", "Kisumu"], 455),
    (["Nairobi", "Nakuru", "Eldoret", "Kakamega", "Kisumu"], 504),
    (["Nairobi", "Nakuru", "Kericho", "Kisii", "Kisumu"], 518),
]


traffic_rules = {
    "Red": "STOP — Engage air brakes smoothly, hold trailer before stop line; unsafe to move.",
    "Yellow": "SLOW DOWN — Downshift and apply gradual braking to prevent fuel sloshing; prepare to stop.",
    "Green": "GO — Safe to proceed within heavy-trailer speed limit (<= 65 km/h).",
}

weather_rules = {
    "Clear": "Normal traction and visibility — maintain standard highway cruise speed.",
    "Rain": "Wet tarmac & slosh risk — reduce speed by 25%, increase following distance, engage wipers/low beams.",
    "Fog": "Low visibility — reduce speed to <= 40 km/h, activate fog & hazard lights, avoid overtaking.",
}

sensor_rules = {
    "Normal": "All tanker sensors nominal (pressure, temperature, roll angle, and vapour levels safe).",
    "VapourLeak": "EMERGENCY SHUT-OFF — Close fuel tank emergency valves, activate hazard alarm, pull off road, alert dispatcher.",
    "HighRollAngle": "STABILITY ALERT — Decelerate immediately on bend/slope to prevent fuel sloshing rollover.",
    "HighTankTemp": "THERMAL ALERT — Pull over in shaded safe bay, inspect tank pressure relief valve, notify control centre.",
    "LowTyrePressure": "TYRE ALERT — Reduce speed to <= 35 km/h and proceed to nearest safe service bay.",
}

# Dynamic facts store (enriched via Open-Meteo API in Step 2 or offline fallback)
town_weather_facts = {town: "Clear" for town in TOWNS}
blocked_corridors = set()  # Stores tuples like ('Nakuru', 'Kisumu')


def load_api_weather_knowledge(local_json_path="kenya_highway_weather_kb.json"):
    """Loads weather facts from fetch_route_kb.py output or Open-Meteo API with safe offline fallback."""
    try:
        with open(local_json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for town, info in data.get("towns", {}).items():
            if town in town_weather_facts:
                town_weather_facts[town] = info.get("condition", "Clear")
        print(f"Loaded weather facts from '{local_json_path}' (Open-Meteo API cache).")
        return
    except FileNotFoundError:
        pass

    # Attempt lightweight live query to Open-Meteo API (permitted, no key required)
    api_success = 0
    for town, (lat, lon) in TOWN_COORDINATES.items():
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        try:
            with urllib.request.urlopen(url, timeout=3) as resp:
                payload = json.loads(resp.read().decode("utf-8"))
                code = int(payload.get("current_weather", {}).get("weathercode", 0))
                if code in (45, 48):
                    town_weather_facts[town] = "Fog"
                elif code >= 51:
                    town_weather_facts[town] = "Rain"
                else:
                    town_weather_facts[town] = "Clear"
                api_success += 1
        except Exception:
            break

    if api_success == len(TOWNS):
        print("Loaded live weather facts for all 7 Kenyan towns via Open-Meteo API.")
    else:
        print("API offline/unreachable — initialized default 'Clear' weather facts for all 7 towns.")


load_api_weather_knowledge()
print(f"Vocabulary & Domain KB initialized: {len(TOWNS)} towns, {sum(len(v) for v in HIGHWAY_GRAPH.values())} directed links, {len(traffic_rules)} base signal rules.")

API offline/unreachable — initialized default 'Clear' weather facts for all 7 towns.
Vocabulary & Domain KB initialized: 7 towns, 17 directed links, 3 base signal rules.


**Step 5: Add New Facts to the Knowledge Base**

In [ ]:
def show_kb():
    """Displays the current Knowledge Base (Traffic Rules, Weather Facts, Sensor Rules, and Highway Graph)."""
    print("\n==================== CURRENT TRAILER KNOWLEDGE BASE ====================")
    print("[1] Traffic Signal Rules:")
    for light, action in traffic_rules.items():
        print(f"    {light:10} -> {action}")

    print("\n[2] Town Weather Facts (from API / Sensor Updates):")
    for town, cond in town_weather_facts.items():
        print(f"    {town:10} -> {cond} ({weather_rules.get(cond, 'Standard caution')})")

    print("\n[3] Tanker Telemetry & Hazard Rules:")
    for status, action in sensor_rules.items():
        print(f"    {status:16} -> {action}")

    print("\n[4] Blocked Highway Corridors:")
    if blocked_corridors:
        for u, v in sorted(blocked_corridors):
            print(f"    {u} -> {v} [CLOSED / UNSAFE]")
    else:
        print("    None (all PE3 highway links open)")
    print("========================================================================")


def add_new_fact(category, key, value):
    """Programmatically TELLs a new fact or rule to the Knowledge Base (without user input)."""
    cat = category.strip().lower()
    if cat == "traffic":
        k = key.strip().capitalize()
        traffic_rules[k] = value.strip()
        print(f"Added traffic rule: '{k}' -> '{traffic_rules[k]}'")
    elif cat == "weather":
        t = key.strip().capitalize()
        c = value.strip().capitalize()
        town_weather_facts[t] = c
        print(f"Updated weather fact: Weather({t}, {c})")
    elif cat == "sensor":
        s = key.strip()
        sensor_rules[s] = value.strip()
        print(f"Added sensor rule: '{s}' -> '{sensor_rules[s]}'")
    elif cat == "blocked":
        u, v = key[0].strip().capitalize(), key[1].strip().capitalize()
        blocked_corridors.add((u, v))
        print(f"Added road closure fact: Blocked({u} -> {v})")
    else:
        print(f"Unknown KB category '{category}'. Use: traffic, weather, sensor, or blocked.")


# Programmatic TELL updates expanding the base domain knowledge
add_new_fact("traffic", "Flashing", "STOP at intersection, verify cross-traffic is clear, then proceed cautiously.")
add_new_fact("traffic", "Blinking", "PROCEED SLOWLY (<= 30 km/h) with hazard lights active.")
add_new_fact("weather", "Kericho", "Rain")
add_new_fact("sensor", "BrakeOverheat", "BRAKE THERMAL ALERT — Engage engine retarder, pull over safely, cool brakes.")


def add_fact():
    """Interactive TELL interface (Option A) to add new knowledge/facts to the KB."""
    print("\n--- Add a New Fact to the Trailer Knowledge Base ---")
    new_light = input("Enter new traffic light signal (or press Enter to add default 'White'): ").strip().capitalize()
    if not new_light:
        new_light = "White"
    new_action = input(f"What should the trailer do when signal is '{new_light}'? (or press Enter for default): ").strip()
    if not new_action:
        new_action = "STOP IMMEDIATELY — Emergency railway/weighbridge override signal."
    traffic_rules[new_light] = new_action
    print(f"New fact added: '{new_light}' -> '{new_action}'")


add_fact()
show_kb()

Added traffic rule: 'Flashing' -> 'STOP at intersection, verify cross-traffic is clear, then proceed cautiously.'
Added traffic rule: 'Blinking' -> 'PROCEED SLOWLY (<= 30 km/h) with hazard lights active.'
Updated weather fact: Weather(Kericho, Rain)
Added sensor rule: 'BrakeOverheat' -> 'BRAKE THERMAL ALERT — Engage engine retarder, pull over safely, cool brakes.'

--- Add a New Fact to the Trailer Knowledge Base ---
Enter new traffic light signal (or press Enter to add default 'White'): White
What should the trailer do when signal is 'White'? (or press Enter for default): STOP IMMEDIATELY — Emergency railway/weighbridge override signal.
New fact added: 'White' -> 'STOP IMMEDIATELY — Emergency railway/weighbridge override signal.'

==================== CURRENT TRAILER KNOWLEDGE BASE ====================
[1] Traffic Signal Rules:
    Red        -> STOP — Engage air brakes smoothly, hold trailer before stop line; unsafe to move.
    Yellow     -> SLOW DOWN — Downshift and apply gradual b

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers**

In [ ]:
def find_optimal_safe_path(start, goal, blocked=None):
    """Uniform-Cost Search over the Practical Exercise 3 Directed Graph respecting blocked edges."""
    if start not in HIGHWAY_GRAPH or goal not in HIGHWAY_GRAPH:
        return None, float("inf")
    blocked = blocked or set()
    frontier = [(0, start, [start])]
    best_cost = {start: 0}

    while frontier:
        cost, current, path = heapq.heappop(frontier)
        if current == goal:
            return path, cost
        if cost > best_cost.get(current, float("inf")):
            continue
        for neighbor, dist in HIGHWAY_GRAPH.get(current, {}).items():
            if (current, neighbor) in blocked:
                continue
            new_cost = cost + dist
            if new_cost < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = new_cost
                heapq.heappush(frontier, (new_cost, neighbor, path + [neighbor]))
    return None, float("inf")


def trailer_decision_agent(current_town, goal_town, observed_light, sensor_status="Normal", temp_blocked=None):
    """ASK engine: infers safe driving action & optimal route from the Trailer Knowledge Base."""
    town = current_town.strip().capitalize()
    dest = goal_town.strip().capitalize()
    light = observed_light.strip().capitalize()
    sensor = sensor_status.strip()
    active_blocked = set(blocked_corridors)
    if temp_blocked:
        active_blocked.add(temp_blocked)

    # 1. Validate locations against PE3 State Space
    if town not in HIGHWAY_GRAPH or dest not in HIGHWAY_GRAPH:
        return {
            "status": "HALT — UNKNOWN LOCATION",
            "decision": f"Hold position and query GPS/dispatcher: '{town}' or '{dest}' is outside the mapped highway network.",
            "route": None,
            "total_km": None,
        }

    # 2. Priority 1: Tanker Telemetry Safety Rule Check
    if sensor != "Normal":
        sensor_action = sensor_rules.get(
            sensor,
            "UNKNOWN SENSOR FAULT — Pull over safely, activate hazard lights, and contact dispatcher."
        )
        return {
            "status": "EMERGENCY / SAFETY OVERRIDE",
            "decision": sensor_action,
            "route": [town],
            "total_km": 0,
        }

    # 3. Priority 2: Traffic Signal Rule Check
    light_action = traffic_rules.get(
        light,
        "UNKNOWN SIGNAL — Stop safely before intersection and proceed only with manual/dispatcher clearance."
    )
    if not light_action.startswith("GO"):
        return {
            "status": "TRAFFIC CONTROL HOLD / CAUTION",
            "decision": light_action,
            "route": [town],
            "total_km": 0,
        }

    # 4. Priority 3: Compute Optimal Safe Path on PE3 Directed Graph
    route, total_km = find_optimal_safe_path(town, dest, active_blocked)
    if not route:
        return {
            "status": "NO SAFE ROUTE AVAILABLE",
            "decision": f"All highway corridors from {town} to {dest} are currently blocked. Hold at safe depot.",
            "route": None,
            "total_km": None,
        }

    # 5. Priority 4: Weather & Speed Advisory along Selected Route
    weather_notes = []
    for node in route:
        cond = town_weather_facts.get(node, "Clear")
        if cond != "Clear":
            weather_notes.append(f"{node}: {cond} ({weather_rules.get(cond)})")
    weather_summary = "; ".join(weather_notes) if weather_notes else weather_rules["Clear"]

    next_hop = route[1] if len(route) > 1 else dest
    step_km = HIGHWAY_GRAPH[town][next_hop] if len(route) > 1 else 0
    decision_msg = (
        f"{light_action} Next leg: {town} -> {next_hop} ({step_km} km). "
        f"Weather advisory: {weather_summary}"
    )
    return {
        "status": "OPTIMAL ROUTE ACTIVE",
        "decision": decision_msg,
        "route": route,
        "total_km": total_km,
    }


# Step 6: Current observations from sensors / operator input (Option A)
obs_town = input("Enter current town (default: Nairobi): ").strip() or "Nairobi"
obs_goal = input("Enter destination town (default: Kisumu): ").strip() or "Kisumu"
obs_light = input("Enter observed traffic light color (default: Green): ").strip() or "Green"
obs_sensor = input("Enter tanker sensor status (default: Normal): ").strip() or "Normal"

# Step 7: Agent queries the Knowledge Base and outputs the decision
result = trailer_decision_agent(obs_town, obs_goal, obs_light, obs_sensor)
print("\n=== Autonomous Fuel Trailer Decision System ===")
print(f"Observed State : At({obs_town.capitalize()}), Goal({obs_goal.capitalize()}), Light({obs_light.capitalize()}), Sensor({obs_sensor})")
print(f"System Status  : {result['status']}")
if result["route"]:
    print(f"Selected Route : {' -> '.join(result['route'])} (Total Distance: {result['total_km']} km)")
print(f"Agent Decision : {result['decision']}")

Enter current town (default: Nairobi): Nairobi
Enter destination town (default: Kisumu): Kisumu
Enter observed traffic light color (default: Green): Green
Enter tanker sensor status (default: Normal): Normal

=== Autonomous Fuel Trailer Decision System ===
Observed State : At(Nairobi), Goal(Kisumu), Light(Green), Sensor(Normal)
System Status  : OPTIMAL ROUTE ACTIVE
Selected Route : Nairobi -> Nakuru -> Kisumu (Total Distance: 339 km)
Agent Decision : GO — Safe to proceed within heavy-trailer speed limit (<= 65 km/h). Next leg: Nairobi -> Nakuru (161 km). Weather advisory: Normal traction and visibility — maintain standard highway cruise speed.


**Step 8: Querying the Knowledge Base**

In [ ]:
# Query 1: Compare all 5 Practical Exercise 3 Search Tree paths from Nairobi to Kisumu
print("--- Query 1: Practical Exercise 3 Search Tree Route Evaluation (Nairobi -> Kisumu) ---")
for route, cost in PE3_SEARCH_TREE_ROUTES:
    tag = "<-- OPTIMAL (PE3)" if cost == 339 else f"(+{cost - 339} km longer)"
    print(f"  {' -> '.join(route):48} | g(n) = {cost} km {tag}")

# Query 2: Dynamic Rerouting when direct Nakuru -> Kisumu highway is closed
print("\n--- Query 2: Dynamic Rerouting when Nakuru -> Kisumu is Blocked ---")
reroute_res = trailer_decision_agent("Nairobi", "Kisumu", "Green", "Normal", temp_blocked=("Nakuru", "Kisumu"))
print("New Optimal Route:", " -> ".join(reroute_res["route"]), f"({reroute_res['total_km']} km)")
print("Agent Decision   :", reroute_res["decision"])

# Query 3: Specific Traffic Signal & Tanker Safety Queries
print("\n--- Query 3: Traffic Light & Tanker Safety Queries ---")
for sig in ["Flashing", "Red", "Teal"]:
    res = trailer_decision_agent("Nakuru", "Kisumu", sig, "Normal")
    print(f"Signal '{sig:8}' -> {res['decision']}")

leak_res = trailer_decision_agent("Nakuru", "Kisumu", "Green", "VapourLeak")
print(f"Sensor 'VapourLeak' -> {leak_res['decision']}")

--- Query 1: Practical Exercise 3 Search Tree Route Evaluation (Nairobi -> Kisumu) ---
  Nairobi -> Nakuru -> Kisumu                      | g(n) = 339 km <-- OPTIMAL (PE3)
  Nairobi -> Nakuru -> Kericho -> Kisumu           | g(n) = 362 km (+23 km longer)
  Nairobi -> Nakuru -> Eldoret -> Kisumu           | g(n) = 455 km (+116 km longer)
  Nairobi -> Nakuru -> Eldoret -> Kakamega -> Kisumu | g(n) = 504 km (+165 km longer)
  Nairobi -> Nakuru -> Kericho -> Kisii -> Kisumu  | g(n) = 518 km (+179 km longer)

--- Query 2: Dynamic Rerouting when Nakuru -> Kisumu is Blocked ---
New Optimal Route: Nairobi -> Nakuru -> Kericho -> Kisumu (362 km)
Agent Decision   : GO — Safe to proceed within heavy-trailer speed limit (<= 65 km/h). Next leg: Nairobi -> Nakuru (161 km). Weather advisory: Kericho: Rain (Wet tarmac & slosh risk — reduce speed by 25%, increase following distance, engage wipers/low beams.)

--- Query 3: Traffic Light & Tanker Safety Queries ---
Signal 'Flashing' -> STOP at intersecti

**Step 9: Debugging the Knowledge Base**

In [ ]:
# Test faulty/unknown traffic signals, unmapped towns, and sensor anomalies
print("--- Debugging Test 1: Known, Unknown & Faulty Traffic Signals ---")
test_signals = ["Red", "Yellow", "Green", "Blue", "Teal", "None", "Purple"]
for color in test_signals:
    outcome = trailer_decision_agent("Nairobi", "Kisumu", color, "Normal")
    print(f"Light: {color:6} -> [{outcome['status']}] {outcome['decision']}")

print("\n--- Debugging Test 2: Edge Cases (Unmapped Town, Sensor Fault, Total Corridor Closure) ---")
edge_cases = [
    ("Mombasa", "Kisumu", "Green", "Normal", None, "Unmapped origin town"),
    ("Nakuru", "Kisumu", "Green", "HighRollAngle", None, "High roll angle on Rift Valley escarpment"),
    ("Nakuru", "Kisumu", "Green", "UnknownSensorX", None, "Unrecognized sensor code"),
    ("Nairobi", "Kisumu", "Green", "Normal", ("Nairobi", "Nakuru"), "Only exit highway Nairobi->Nakuru closed"),
]
for town, goal, light, sensor, blocked_edge, label in edge_cases:
    res = trailer_decision_agent(town, goal, light, sensor, temp_blocked=blocked_edge)
    print(f"Case ({label}):\n   -> [{res['status']}] {res['decision']}")

--- Debugging Test 1: Known, Unknown & Faulty Traffic Signals ---
Light: Red    -> [TRAFFIC CONTROL HOLD / CAUTION] STOP — Engage air brakes smoothly, hold trailer before stop line; unsafe to move.
Light: Yellow -> [TRAFFIC CONTROL HOLD / CAUTION] SLOW DOWN — Downshift and apply gradual braking to prevent fuel sloshing; prepare to stop.
Light: Green  -> [OPTIMAL ROUTE ACTIVE] GO — Safe to proceed within heavy-trailer speed limit (<= 65 km/h). Next leg: Nairobi -> Nakuru (161 km). Weather advisory: Normal traction and visibility — maintain standard highway cruise speed.
Light: Blue   -> [TRAFFIC CONTROL HOLD / CAUTION] UNKNOWN SIGNAL — Stop safely before intersection and proceed only with manual/dispatcher clearance.
Light: Teal   -> [TRAFFIC CONTROL HOLD / CAUTION] UNKNOWN SIGNAL — Stop safely before intersection and proceed only with manual/dispatcher clearance.
Light: None   -> [TRAFFIC CONTROL HOLD / CAUTION] UNKNOWN SIGNAL — Stop safely before intersection and proceed only with man

Debugging confirms that whenever the agent encounters an unrecognized signal (such as `Blue`, `Teal`, or `None`), an unmapped town (`Mombasa`), an unknown sensor fault, or a complete highway closure (`Nairobi -> Nakuru`), it defaults to a **fail-safe stop/hold action** rather than guessing. When new legitimate signals or road links are verified (for example, adding a `Narok` bypass or a `None` = *treat unpowered traffic light as a 4-way stop* rule), they can be inserted immediately using `add_new_fact(...)`.

**Step 10: Add final reflection here (challenges or recommended improvements)**

### Challenges Encountered
1. **Combining Symbolic Rules with Graph Search:** Practical Exercise 3 focused purely on static shortest-path search ($g(n) = 339\text{ km}$ via `Nairobi -> Nakuru -> Kisumu`), whereas Practical Exercise 4 required real-time logical reasoning over dynamic percepts (traffic signals, weather, and tanker safety sensors). Designing a clean priority hierarchy—where tanker safety and traffic laws override route execution—was essential to avoid unsafe decisions.
2. **Partial Observability & Sensor Uncertainty:** As identified in our PEAS environment classification, Kenya's highways are partially observable and stochastic. Propositional rules (`Weather(Kericho, Rain)`) assume accurate sensor/API inputs, whereas real sensors can experience noise or missing readings.
3. **External API Dependency vs. Offline Reliability:** Querying the Open-Meteo API enriches the Knowledge Base with live meteorological facts, but cellular dead zones along Rift Valley escarpments can cause request timeouts. Implementing a cached JSON loader (`fetch_route_kb.py`) with an automatic offline fallback solved this reliability gap.

### Recommended Improvements
1. **First-Order Logic (FOL) & Forward/Backward Chaining Engine:** Replace flat dictionary lookups with a formal Horn-clause inference engine (e.g., `Connected(x, y) ^ Weather(y, Rain) -> ReduceSpeed(Trailer, x, y)`) to scale cleanly across all 47 Kenyan counties without duplicating rules.
2. **Informed $A^*$ Search with Haversine Heuristics:** Use the GPS coordinates already encoded in `TOWN_COORDINATES` to compute straight-line distances $h(n)$ to Kisumu alongside actual road distances $g(n)$, improving search efficiency on larger national road networks.
3. **Probabilistic Reasoning (Bayesian Networks / Markov Decision Processes):** Integrate confidence weights for noisy camera/vapour sensors and stochastic traffic delays so the trailer can reason under uncertainty.